In [30]:
# %% Minimal setup from class
import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "bens_model")              

SYSTEM_PROMPT = "You are a helpful assistant with limited text output. Reply with only the final answer—no explanation. There is no need to repeat the question."
TEMPERATURE   = 0.25 #Must be a float

def call_model_chat_completions(prompt: str,
                                system: str = SYSTEM_PROMPT,
                                model: str = MODEL,
                                temperature: float = TEMPERATURE,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 350,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)

def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        #1) Get model prediction
        if t.get("input"):
            r = call_model_chat_completions(
                t["input"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )
            #got = (r.get("text") or "").strip()
            got = agent_loop(t["input"])
            
            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["input"],
                prediction=got,
                expected_answer=t["output"],
                model=judge_model,
            )
        else:
            r = call_model_chat_completions(
                t["prompt"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )        
            #got = (r.get("text") or "").strip()    
            got = agent_loop(t["prompt"])

            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["prompt"],
                prediction=got,
                expected_answer=t["expected"],
                model=judge_model,
            )



        row = {
            "id": t.get("id", "<unnamed>"),
            "output": t["output"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: output={row['output']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows


In [31]:
def reasoning_via_planning(prior:str, question: str,) -> dict:
    prior_reasoning = "\nPrior Reasoning: " + prior + "\n\n"
    reasoning_str = "Create a plan using as little words as possible. Using prior reasoning, decompose the problem into a few step to solve the question provided. Execute each step in order to arrive at the final answer.If the question involves math, write a python program that solves the question and exports an answer. Make sure your answer solves the question provided.\n\n Question: "

    r = call_model_chat_completions(
            reasoning_str + question + prior_reasoning,
            system="You are a planner. Provide a short, structured step-by-step plan to solve the question.",
            model=MODEL,
            temperature=0.35,
        )
    got = (r.get("text") or "").strip()
    return got

In [32]:
def tree_of_thought(question: str, n_paths: int, prior: str = None,):
    tot_str = "You will decompose the problem down into {n_paths} distinct possible solution paths to solve the question provided. Depending on the problem, write a short reasonable path that is different from every other path created but still leads to the answer. Expected output should be in the form of: path1:<>, \npath2:<>,etc.\n\n Question: "
    r = call_model_chat_completions(
            tot_str.format(n_paths=n_paths) + question,
            system="You are a problem solver that creates multiple distinct solution paths to solve a problem. Use as little words as possible.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    #Divide into n_paths
    raw = r.get("text") or ""
    thoughts = []
    for n in range(1, n_paths + 1):
        path = raw.find(f"path{n}:")
        end = raw.find(f"path{n+1}:")
        if end == -1:
            end = len(raw)
        thoughts.append(raw[path:end].strip())
    return thoughts

In [33]:
def self_consistency(input_question: str, n_paths: int, prior: str = "",):
    paths = []
    for n in range(1, n_paths + 1):
        r = call_model_chat_completions(
                prior + input_question,
                system="",
                model=MODEL,
                temperature=TEMPERATURE,
            )
        got = (r.get("text") or "").strip()
        paths.append(got)
    return paths

In [34]:
def double_check(prior:str, question: str):
    original_question = "Original Question: " + question + "\n"
    check_str = "Verify this solution.If incorrect, output the correct solution. Output ONLY the final answer in the exact required format. No explanations, no markdown, no labels."
    r = call_model_chat_completions(
            original_question + check_str + prior,
            system="You verify solutions and output ONLY the final answer. No 'Answer:', no bold **, no quotes, no extra text. Just the raw answer value.",
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

In [35]:
import requests, trafilatura
from urllib.parse import quote
# Returns text content of most relevant Wikipedia page for a question
def wiki_tool(input_question: str, attempts=1):
    attempted_titles = "None"
    for i in range(attempts):
        r = call_model_chat_completions(
                prompt = f'''Given a question, provide the title of the most relevant Wikipedia page that answers the question. 
                Only provide the title, no explanations.
                Examples: 
                Question: Which genus of moth in the world's seventh-largest country contains only one species?
                Answer: Crambidae
                Question: What U.S Highway gives access to Zilpo Road, and is also known as Midland Trail?
                Answer: US 60
                
                Do NOT use these titles if they have been tried already: {attempted_titles}
                Question: {input_question}''',
                system="You are a Wikipedia search assistant. Given a question, return the title of the most relevant Wikipedia page that answers the question. Reply with only the title, no explanations.",
                model=MODEL,
                temperature=0.2,
            )
        title = (r.get("text") or "").strip()
        attempted_titles = attempted_titles + "[" + title + "],"
        
        try:
            #Calls a wiki API to get page of a topic
            url = f"https://en.wikipedia.org/api/rest_v1/page/mobile-html/{quote(title)}"
            headers = {"User-Agent": "MilkBot/1.0 (https://github.com/Isaiah-Milkey)", "Accept": "text/html"}
            r = requests.get(url, headers=headers, timeout=20)
            r.raise_for_status()

            extracted = trafilatura.extract(
                r.text,
                include_tables=True,
                include_comments=False,
                output_format="txt"  
            )
            extracted = f"{title}: " + extracted
        except Exception as e:
            #print(f"Error fetching Wikipedia page for title '{title}': {e}")
            extracted = f"No extractable content found for article: {title}."
    return extracted or "No Wikipedia content found."

In [36]:
#Following example from: https://github.com/paaxel/llama-starter-examples/blob/main/6-hello-llama-rag.ipynb
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
import json

data = []
with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

data = []
for test in DEV_DATA:
    data.append(
        Document(
            page_content = str(test["output"]),
            metadata = {"input": test["input"], "Domain": test["domain"]}
        )
    )

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = HuggingFaceEmbeddings(model_name=embedding_model_name)
vector_store = FAISS.from_documents(data, embedding_model)

def RAG_tool(question: str, num_examples: int = 1):
    query = vector_store.similarity_search(question, k=num_examples)
    
    content = ""
    for i, doc in enumerate(query, 1):
        content += f"Example {i}:\n"
        content += f"Input: {doc.metadata['input']}\n"
        content += f"Output: {doc.page_content}\n\n"
    
    return content.strip()


In [37]:
def calc_tool(input_question: str):
    retrieved_examples = RAG_tool(input_question, num_examples=2)


    MATH_AGENT_PROMPT = f"""
    You MUST respond in exactly one of these two formats: 
    Arithmetic: <arithmetic expression>
    FINAL: <answer>  

    - use only numbers, + - * / **, parentheses, and round(x, ndigits)
    - avoid using parentheses withing parentheses where possible AND always close open parentheses. eg: (((2*3)*4)*5)+1
    Example of Arithmetic: round((3*2.49)*1.07, 2)
    Example of FINAL: FINAL: 23
    Return ONE line with NO explanations. No other text.
    Examples from similar problems:
    {retrieved_examples}
    """
    r = call_model_chat_completions(
            prompt= f"""Question: {input_question}
            If you need arithmetic to solve the question, reply as:
            <expression>
            
            Otherwise reply:
            FINAL: <answer> """,
            system=MATH_AGENT_PROMPT,
            model=MODEL,
            temperature=0,
        )

    case = (r.get("text") or "").strip().lower()
    
    if "arithmetic:" in case:
        try:
            expression = case.lstrip("arithmetic:")
            if expression.count("(") > expression.count(")"):
                expression += ")"
            elif expression.count("(") > expression.count(")"):
                expression = "(" + expression
            return eval(expression)
        except:
            return case
    else:
        return case

In [38]:
# Case definitions for each case: math, coding, future_prediction, planning, and common_sense

#Code inspired by Mini Lab 5 and this youtube video: https://www.youtube.com/watch?v=Uz7pWszyi6k
def case_math(input_question: str):
    #print("entered case_math function")
    RAG_content = RAG_tool(input_question, num_examples=2)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning("For solving math problems, I can create a simple python program that solves the question- and outputs the answer.", prior)
    curr = calc_tool(input_question+curr)
    final = double_check(str(curr), input_question)
    return final

def case_coding(input_question: str):
    RAG_content = RAG_tool(input_question, num_examples=2)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning("For solving coding problems, I can create a simple python program that solves the question- and outputs the answer.", prior)
    curr = double_check(curr, input_question) #Get an output, then perform RAG search on the solution and repeat

    RAG_content = RAG_tool(curr, num_examples=5)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning("For solving coding problems, I can create a simple python program that solves the question- and outputs the answer.", prior)
    r = call_model_chat_completions(
            prompt= f"""Solve this provided problem by creating a python script using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {curr}
            """,
            system="""You are a coding agent that writes simple and reliable code that will always execute correctly.
            You MUST respond using this format:
            Python script: <answer> 
            
            Example: x_values = np.linspace(0, 2 * np.pi, 400)\n    fig, axs = plt.subplots(2)\n
            Example: combined_matrix = np.concatenate((matrix1, matrix2), axis=1)\n    df = pd.DataFrame(combined_matrix)\n    return df.to_string(index=False, header=False)""",
            model=MODEL,
            temperature=0.2,
        )
    final = (r.get("text") or "").strip()
    return final

def case_common_sense(input_question: str):
    RAG_content = RAG_tool(input_question, num_examples=2)
    prior = RAG_content
    curr = prior + wiki_tool(input_question=input_question, attempts=5)
    r = call_model_chat_completions(
            prompt= f"""Solve this provided problem using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {curr}
            """,
            system="""You are a common sense agent that answers questions reasonably and accurately.
            You MUST respond using this format:
            Response: <answer> 
            
            Example:PROBLEM: What American country music singer-songwriter, born in May of 1942, sang a duet with her ex-husband the same year that he released the song The Battle? Response: Tammy Wynette.
            Example: PROBLEM: The first credit cards were for use in what type of establishments? Response: Restaurants.""",
            model=MODEL,
            temperature=0.2,
        )
    final = (r.get("text") or "").strip()
    return final

def case_future_prediction(input_question: str):
    RAG_content = RAG_tool(input_question, num_examples=4)
    prior = RAG_content
    curr = reasoning_via_planning(f"For solving future prediction problems, I can use previous examples to create a prediction for the question: {input_question}\n.", prior)
    curr = double_check(curr, input_question) #Get an output, then perform RAG search on the solution and repeat

    RAG_content = RAG_tool(curr, num_examples=6)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning(f"For solving future prediction problems, I can use previous examples to create a prediction for the question: {input_question}\n.", prior)
    
    r = call_model_chat_completions(
            prompt= f"""Solve the provided problem by creating a prediction using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {curr}
            """,
            system="""You are a future prediction agent that writes reliable predictions that are realistic and accurate.
            You MUST respond using this format:
            Prediction: <answer> 
            
            Example: PROBLEM: You are an agent that can predict future events. The event to be predicted: \"Starship flight 10 soft touchdown? (around 2025-07-30T23:14:00Z). Starship flight 10 soft touchdown?\"\n        IMPORTANT: Your final answer MUST end with this exact format:\n        \\boxed{Yes} or \\boxed{No}\n        Do not use any other format. Do not refuse to make a prediction. Do not say \"I cannot predict the future.\" You must make a clear prediction based on the best data currently available, using the box format specified above. Prediction: ['No']
            Example: PROBLEM: You are an agent that can predict future events. The event to be predicted: \"\u8bf7\u9884\u6d4b\u5317\u4eac\u65f6\u95f42025-08-08, Steam\u516c\u5e03\u7684Support Stats\u4e2d\uff0c\u7b49\u5f85\u54cd\u5e94\uff08Waiting for response\uff09\u7684\u7b2c\u4e00\u4e2a\u6570\u5b57\u662f\u591a\u5c11\uff1f\"\n        IMPORTANT: Your final answer MUST end with this exact format:\n        \\boxed{YOUR_PREDICTION}\n        Do not use any other format. Do not refuse to make a prediction. Do not say \"I cannot predict the future.\" You must make a clear prediction based on the best data currently available, using the box format specified above. Prediction: [33489.0]""",
            model=MODEL,
            temperature=0.2,
        )
    final = (r.get("text") or "").strip()
    return final

def case_planning(input_question: str):
    RAG_content = RAG_tool(input_question, num_examples=4)
    prior = RAG_content
    curr = reasoning_via_planning(f"For solving future planning problems, I can use previous examples to create a plan for the question: {input_question}\n.", prior)
    curr = double_check(curr, input_question) #Get an output, then perform RAG search on the solution and repeat

    RAG_content = RAG_tool(curr, num_examples=8)
    prior = RAG_content + "\nSolve the following question:\n" + input_question
    curr = reasoning_via_planning(f"For solving future planning problems, I can use previous examples to create a plan for the question: {input_question}\n.", prior)
    curr = self_consistency(input_question, 2, curr)
    curr = "Path 1:"+ curr[0] + "Path 2:" + curr[1]
    r = call_model_chat_completions(
            prompt= f"""Solve the provided problem by creating a plan using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {curr}
            """,
            system="""You are a planning agent that writes reliable plans that are realistic and accurate.
            You MUST respond using this format:
            Plan: <answer> 
            
            Example: PROBLEM: I am playing with a set of objects. Here are the actions I can do\n\n   Attack object\n   Feast object from another object\n   Succumb object\n   Overcome object from another object\n\nI have the following restrictions on my actions:\n    To perform Attack action, the following facts need to be true: Province object, Planet object, Harmony.\n    Once Attack action is performed the following facts will be true: Pain object.\n    Once Attack action is performed the following facts will be false: Province object, Planet object, Harmony.\n    To perform Succumb action, the following facts need to be true: Pain object.\n    Once Succumb action is performed the following facts will be true: Province object, Planet object, Harmony.    \n    Once Succumb action is performed the following facts will be false: Pain object.\n    To perform Overcome action, the following needs to be true: Province other object, Pain object.\n    Once Overcome action is performed the following will be true: Harmony, Province object, Object Craves other object.\n    Once Overcome action is performed the following will be false: Province other object, Pain object.\n    To perform Feast action, the following needs to be true: Object Craves other object, Province object, Harmony.\n    Once Feast action is performed the following will be true: Pain object, Province other object.\n    Once Feast action is performed the following will be false:, Object Craves other object, Province object, Harmony.\n\n[STATEMENT]\nAs initial conditions I have that, object a craves object b, object c craves object d, harmony, planet object b, planet object d, province object a and province object c.\nMy goal is to have that object b craves object d, object c craves object a and object d craves object c.\n\nMy plan is as follows:\n\n[PLAN]\nfeast object a from object b\nsuccumb object a\nfeast object c from object d\novercome object c from object a\nattack object d\novercome object d from object c\nattack object b\novercome object b from object d\n[PLAN END]\n\n[STATEMENT]\nAs initial conditions I have that, object a craves object c, object b craves object d, harmony, planet object c, planet object d, province object a and province object b.\nMy goal is to have that object b craves object c, object c craves object d and object d craves object a.\n\nMy plan is as follows:\n\n[PLAN] Plan: (feast a c)\n(succumb a)\n(feast b d)\n(succumb b)\n(attack d)\n(overcome d a)\n(attack c)\n(overcome c d)\n(attack b)\n(overcome b c)\n
            Example: PROBLEM: I have to plan the logistics of transporting crates between a number of depots and distributors via trucks that are loaded by hoists. Depots and distributors are directly connected by roads (trucks can drive between any two depots or distributors).\n\nA depot is a type of place.\nA distributor is a type of place.\nA pallet is a type of surface.\nA crate is a type of surface.\n\nHere are the actions that can be performed:\n\nDrive a truck from one place to another place.\nUse a hoist to lift a crate from a surface at a place.\nUse a hoist to drop a crate to a surface at a place.\nUse a hoist to load a crate into a truck at a place.\nUse a hoist to unload a crate from a truck at a place.\n\nThe following are the restrictions on the actions:\nA truck can be driven from one place to another place only if the truck is at the origin place.\nOnce a truck has been driven from one place to another, it is not at the origin place and is at the destination place.\nA crate can be lifted by a hoist only if the hoist is at the same place as the crate, the hoist is available, and the crate is clear.\nOnce a crate has been lifted by a hoist from a surface at a place, the crate is not at the place, the hoist is lifting the crate, the hoist is not available, the surface is clear, and the crate is not on the surface.\nA crate can be dropped by a hoist to a surface only if the hoist and surface are both at the place, the surface is clear, and the hoist is lifting the crate.\nOnce a crate has been dropped by a hoist to a surface at a place, the hoist is available, the hoist is not lifting the crate, the crate is at the place, the surface is not clear, the crate is clear, and the crate is on the surface.\nA crate can be loaded by a hoist onto a truck at a place only if the hoist is at the same place, the truck is at the same place, and the hoist is lifting the crate.\nOnce a crate has been loaded by a hoist onto a truck at a place, \nA crate can be unloaded by a hoist from a truck at a place only if the hoist is at the same place as the truck, the hoist is available, and the crate is in the truck.\nOnce a crate has been unloaded by a hoist from a truck at a place, the crate is not in the truck, the hoist is not available, and the hoist is lifting the crate.\n\n[STATEMENT]\nAs initial conditions I have that, crate0 is at distributor0, crate1 is at depot2, crate2 is at depot2, hoist0 is at depot0, hoist1 is at depot1, hoist2 is at depot2, hoist3 is at distributor0, pallet0 is at depot0, pallet1 is at depot1, pallet2 is at depot2, pallet3 is at distributor0, truck0 is at depot2, truck1 is at depot1, truck2 is at depot1, hoist0 is available, hoist1 is available, hoist2 is available, hoist3 is available, crate0 is clear, crate2 is clear, pallet0 is clear, pallet1 is clear, crate0 is on pallet3, crate1 is on pallet2 and crate2 is on crate1.\nMy goal is to have that crate0 is on pallet1, crate1 is on crate0 and crate2 is on pallet2.\n\nMy plan is as follows:\n\n[PLAN]\nUse hoist2 to lift crate2 from crate1 at depot2\nUse hoist2 to load crate2 into truck0 at depot2\nUse hoist2 to lift crate1 from pallet2 at depot2\nUse hoist2 to load crate1 into truck0 at depot2\nUse hoist2 to unload crate2 from truck0 at depot2\nUse hoist2 to drop crate2 to pallet2 at depot2\nUse hoist3 to lift crate0 from pallet3 at distributor0\ndrive truck0 from depot2 to distributor0\nUse hoist3 to load crate0 into truck0 at distributor0\ndrive truck0 from distributor0 to depot1\nUse hoist1 to unload crate0 from truck0 at depot1\nUse hoist1 to drop crate0 to pallet1 at depot1\nUse hoist1 to unload crate1 from truck0 at depot1\nUse hoist1 to drop crate1 to crate0 at depot1\n[PLAN END]\n\n[STATEMENT]\nAs initial conditions I have that, crate0 is at distributor0, crate1 is at depot0, crate2 is at depot1, hoist0 is at depot0, hoist1 is at depot1, hoist2 is at depot2, hoist3 is at distributor0, pallet0 is at depot0, pallet1 is at depot1, pallet2 is at depot2, pallet3 is at distributor0, truck0 is at depot2, truck1 is at depot0, truck2 is at depot0, hoist0 is available, hoist1 is available, hoist2 is available, hoist3 is available, crate0 is clear, crate1 is clear, crate2 is clear, pallet2 is clear, crate0 is on pallet3, crate1 is on pallet0 and crate2 is on pallet1.\nMy goal is to have that crate0 is on pallet0, crate1 is on pallet1 and crate2 is on crate1.\n\nMy plan is as follows:\n\n[PLAN] Plan:(drive truck2 depot0 distributor0)\n(lift hoist0 crate1 pallet0 depot0)\n(lift hoist3 crate0 pallet3 distributor0)\n(load hoist3 crate0 truck2 distributor0)\n(drive truck2 distributor0 depot0)\n(load hoist0 crate1 truck2 depot0)\n(unload hoist0 crate0 truck2 depot0)\n(drive truck2 depot0 depot1)\n(drop hoist0 crate0 pallet0 depot0)\n(lift hoist1 crate2 pallet1 depot1)\n(load hoist1 crate2 truck2 depot1)\n(unload hoist1 crate1 truck2 depot1)\n(drop hoist1 crate1 pallet1 depot1)\n(unload hoist1 crate2 truck2 depot1)\n(drop hoist1 crate2 crate1 depot1)\n""",
            model=MODEL,
            temperature=0.2,
        )
    final = (r.get("text") or "").strip()
    return final

In [ ]:
# Good vid for reference: https://www.youtube.com/watch?v=ahnGLM-RC1Y

def agent_loop(input_question: str):
    #Have the model decide on a strategy to solve the problem
    r = call_model_chat_completions(
            "Decide what type of category this question falls into: math, coding, future_prediction, planning, common_sense, or other. Respond with only the category name." + "\n\n Question: " + input_question,
            system="You are a trained classifier agent that matches questions with a category. Lean into selecting math, coding, future_prediction, planning, common_sense, when possible. If the question does not fall under ANY of these categories, then select 'other'",
            model=MODEL,
            temperature=0,
        )
    case = (r.get("text") or "").strip().lower()
    case = case.split()[0] if case.split() else "other"
    print(f"Case decided: {case}\n")
    if case == "math":
        #Math/Coding solving strategy
        print("Using math solving strategy\n")
        return case_math(input_question)
    
    elif case == "coding":
        #Coding solving strategy
        print("Using coding solving strategy\n")
        return case_coding(input_question)
    
    elif case == "common_sense":
        #Use wiki tool
        print("Using common_sense solving strategy with wiki tool\n")
        return case_common_sense(input_question)
    
    elif case == "future_prediction":
        #Use RAG
        print("Using furture_prediction solving strategy with RAG tool")
        return case_future_prediction(input_question)
    
    elif case == "planning":
        #Use reasoning via planning / RAG / Consistency check
        print("Using reasoning via planning solving strategy\n")
        return case_planning(input_question)
    
    elif case == "other":
        #Fallback to self consistency
        print("Using other case: self consistency solving strategy\n")
        RAG_content = RAG_tool(input_question, num_examples=3)
        prior = RAG_content + "\nSolve the following question:\n" + input_question
        paths = self_consistency(prior, n_paths=2)
        combined = "These are the different solution paths: \n"
        for p in paths:
            combined += p + "\n"
        r = call_model_chat_completions(
            prompt= f"""Solve this provided problem using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {combined}
            """,
            system="""You are a helpful agent that writes simple and reliable answers to problems.
            You MUST respond using this format:
            answer: <answer> 
            """,
            model=MODEL,
            temperature=0.2,
        )
        final = (r.get("text") or "").strip()
        return final    
        
    else:
        print("Did not catch a case: Using self consistency\n")
        RAG_content = RAG_tool(input_question, num_examples=3)
        prior = RAG_content + "\nSolve the following question:\n" + input_question
        paths = self_consistency(prior, n_paths=2)
        combined = "These are the different solution paths: \n"
        for p in paths:
            combined += p + "\n"
        r = call_model_chat_completions(
            prompt= f"""Solve this provided problem using the provided reference.
            PROBLEM: {input_question}
            REFERENCE: {combined}
            """,
            system="""You are a helpful agent that writes simple and reliable answers to problems.
            You MUST respond using this format:
            answer: <answer> 
            """,
            model=MODEL,
            temperature=0.2,
        )
        final = (r.get("text") or "").strip()
        return final
        

In [40]:
import json
import random

with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

#Get test batches by domain/random
def filter_domain(domain: str):
    filtered = []
    for test in DEV_DATA:
        if test.get("domain") == domain:
            filtered.append(test)
    return filtered

def get_batch(num: int, domain: str = None, is_random: bool = False):
    #random.seed(315)
    if domain:
        data = filter_domain(domain)
    else:
        data = DEV_DATA

    if is_random:
        return random.sample(data, num)
    else:
        return data[:num]

In [41]:

# Testing the Agent Loop:

#tests = get_batch(10, domain="planning", is_random=True)
#self_evaluate_tests(tests, model=MODEL, sleep_sec=0.5, verbose=True)


In [42]:
#!/usr/bin/env python3
"""
Generate a placeholder answer file that matches the expected auto-grader format.

Replace the placeholder logic inside `build_answers()` with your own agent loop
before submitting so the ``output`` fields contain your real predictions.

Reads the input questions from cse_476_final_project_test_data.json and writes
an answers JSON file where each entry contains a string under the "output" key.
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List


INPUT_PATH = Path("cse_476_final_project_test_data.json")
OUTPUT_PATH = Path("cse_476_final_project_answers.json")


def load_questions(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as fp:
        data = json.load(fp)
    if not isinstance(data, list):
        raise ValueError("Input file must contain a list of question objects.")
    return data


def build_answers(questions: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    answers = []
    for idx, question in enumerate(questions, start=1):
        # Example: ass0ume you have an agent loop that produces an answer string.
        # real_answer = agent_loop(question["input"])
        # answers.append({"output": real_answer})
        answer = agent_loop(question["input"])
        answers.append({"output": answer})
    return answers

def build_answers2(questions: List[Dict[str, Any]]):
    total = len(questions)

    with OUTPUT_PATH.open("w", encoding="utf-8") as fp:
        fp.write("[\n")

        for idx, question in enumerate(questions):
            answer = agent_loop(question["input"])
            entry = {"output": answer}

            #Write entry with proper comma placement
            json.dump(entry, fp, ensure_ascii=False, indent=2)

            if idx < len(questions) - 1:
                fp.write(",\n")
            else:
                fp.write("\n")
            print(f"[{idx + 1}/{total}] Finished question {idx + 1}")

        fp.write("]\n")

def validate_results(
    questions: List[Dict[str, Any]], answers: List[Dict[str, Any]]
) -> None:
    if len(questions) != len(answers):
        raise ValueError(
            f"Mismatched lengths: {len(questions)} questions vs {len(answers)} answers."
        )
    for idx, answer in enumerate(answers):
        if "output" not in answer:
            raise ValueError(f"Missing 'output' field for answer index {idx}.")
        if not isinstance(answer["output"], str):
            raise TypeError(
                f"Answer at index {idx} has non-string output: {type(answer['output'])}"
            )
        if len(answer["output"]) >= 5000:
            raise ValueError(
                f"Answer at index {idx} exceeds 5000 characters "
                f"({len(answer['output'])} chars). Please make sure your answer does not include any intermediate results."
            )



questions = load_questions(INPUT_PATH)
answers = build_answers2(questions)

Case decided: other

Using other case: self consistency solving strategy

[1/6208] Finished question 1
Case decided: other

Using other case: self consistency solving strategy

[2/6208] Finished question 2
Case decided: other

Using other case: self consistency solving strategy

[3/6208] Finished question 3
Case decided: other

Using other case: self consistency solving strategy

[4/6208] Finished question 4
Case decided: other

Using other case: self consistency solving strategy

[5/6208] Finished question 5
Case decided: other

Using other case: self consistency solving strategy

[6/6208] Finished question 6
Case decided: other

Using other case: self consistency solving strategy

[7/6208] Finished question 7
Case decided: other

Using other case: self consistency solving strategy

[8/6208] Finished question 8
Case decided: other

Using other case: self consistency solving strategy

[9/6208] Finished question 9
Case decided: other

Using other case: self consistency solving strategy


KeyboardInterrupt: 

In [ ]:
#Streamed input check
with OUTPUT_PATH.open("r", encoding="utf-8") as fp:
    raw = fp.read().strip()

#Check if current JSON ends with "]")
if not raw.endswith("]"):
    print("JSON file is incomplete")
else:
    saved_answers = json.loads(raw)
    validate_results(questions, saved_answers)
    print(
        f"Wrote {len(saved_answers)} answers to {OUTPUT_PATH} and is correct."
    )

# with OUTPUT_PATH.open("w", encoding="utf-8") as fp:
#     json.dump(answers, fp, ensure_ascii=False, indent=2)

# with OUTPUT_PATH.open("r", encoding="utf-8") as fp:
#      saved_answers = json.load(fp)
# validate_results(questions, saved_answers)
# print(
#     f"Wrote {len(answers)} answers to {OUTPUT_PATH} "
#     "and validated format successfully."
# )

Wrote 3 answers to cse_476_final_project_answers.json and validated format successfully.
